In [0]:
import requests
import json
from datetime import datetime, timezone, timedelta
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType, ArrayType
import time
import uuid
import hashlib

In [0]:
# Catalog Configuration
CATALOG = "workspace"  # Update with your catalog

# Schema Names - following LMIP architecture
BRONZE_SCHEMA = "bronze"      # Raw API snapshots
METADATA_SCHEMA = "metadata"  # Pipeline run control
AUDIT_SCHEMA = "audit"        # Pipeline audit logs

# Bronze Tables (raw ingestion layer)
BRONZE_JOB_SNAPSHOT = f"{CATALOG}.{BRONZE_SCHEMA}.bronze_job_snapshot"
BRONZE_API_LOG = f"{CATALOG}.{BRONZE_SCHEMA}.bronze_api_response_log"

# Metadata Tables
PIPELINE_RUN_CONTROL = f"{CATALOG}.{METADATA_SCHEMA}.pipeline_run_control"
SOURCE_CONFIG = f"{CATALOG}.{METADATA_SCHEMA}.source_config"

# Audit Tables
AUDIT_PIPELINE_RUNS = f"{CATALOG}.{AUDIT_SCHEMA}.audit_pipeline_runs"

# Define explicit schema for bronze job snapshot
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType, DateType

BRONZE_SNAPSHOT_SCHEMA = StructType([
    StructField("snapshot_id", StringType(), False),
    StructField("source_name", StringType(), False),
    StructField("source_job_id", StringType(), True),
    StructField("batch_id", StringType(), False),
    StructField("page_number", IntegerType(), True),
    StructField("page_size", IntegerType(), True),
    StructField("payload_json", StringType(), False),
    StructField("payload_hash", StringType(), False),
    StructField("ingestion_timestamp", TimestampType(), False),
    StructField("ingestion_date", DateType(), False),
    StructField("source_status_code", IntegerType(), True),
    StructField("source_etag", StringType(), True)
])

In [0]:
def generate_snapshot_id(source_name, source_job_id, batch_id):
    """Generate unique snapshot ID for bronze layer"""
    composite = f"{source_name}:{source_job_id}:{batch_id}"
    return hashlib.sha256(composite.encode()).hexdigest()[:32]

def generate_batch_id():
    """Generate unique batch ID for ingestion run"""
    return str(uuid.uuid4())

def generate_payload_hash(payload_json):
    """Generate hash of payload for deduplication"""
    return hashlib.sha256(payload_json.encode()).hexdigest()

In [0]:
def process_to_bronze(jobs_data, source_name, batch_id, extract_job_id_func):
    """Process API response to bronze layer format
    
    Args:
        jobs_data: List of job records from API
        source_name: Source identifier (arbeitnow, remotive, etc.)
        batch_id: Unique batch identifier
        extract_job_id_func: Source-specific function to extract job ID from record
    """
    if not jobs_data:
        return []
    
    bronze_records = []
    ingestion_timestamp = datetime.now(timezone.utc)
    ingestion_date = ingestion_timestamp.date()
    
    for job in jobs_data:
        # Use source-specific extraction function
        source_job_id = extract_job_id_func(job)
        
        payload_json = json.dumps(job)
        
        bronze_records.append({
            'snapshot_id': generate_snapshot_id(source_name, source_job_id, batch_id),
            'source_name': source_name,
            'source_job_id': source_job_id,
            'batch_id': batch_id,
            'page_number': None,
            'page_size': None,
            'payload_json': payload_json,
            'payload_hash': generate_payload_hash(payload_json),
            'ingestion_timestamp': ingestion_timestamp,
            'ingestion_date': ingestion_date,
            'source_status_code': 200,
            'source_etag': None
        })
    
    return bronze_records

In [0]:
def ingest_jobs(source_name, fetch_function, api_url, extract_job_id_func,
                log_api_response_func, start_pipeline_run_func, 
                complete_pipeline_run_func, log_audit_func):
    """Main ingestion function following LMIP bronze layer pattern"""
    pipeline_name = f"bronze_ingestion_{source_name}"
    batch_id = generate_batch_id()
    start_time = time.time()
    
    # Start pipeline run in metadata
    run_control_sk = start_pipeline_run_func(pipeline_name, source_name, batch_id)
    print(f"\nPipeline: {pipeline_name}")
    print(f"Batch ID: {batch_id}")
    
    try:
        # Fetch data from API
        print(f"\n[1/3] Fetching data from {source_name}...")
        api_start = time.time()
        jobs_data, error = fetch_function()
        api_duration_ms = int((time.time() - api_start) * 1000)
        
        if error:
            # Log failed API response
            log_api_response_func(source_name, batch_id, api_url, 500, 
                           response_time_ms=api_duration_ms)
            
            duration = time.time() - start_time
            complete_pipeline_run_func(batch_id, 'FAILED')
            log_audit_func(batch_id, pipeline_name, 'FAILED', 
                          runtime_seconds=duration, error_message=error)
            print(f"  FAILED to fetch: {error}")
            return False
        
        records_fetched = len(jobs_data) if jobs_data else 0
        print(f"  Fetched {records_fetched} jobs in {api_duration_ms}ms")
        
        # Log successful API response
        log_api_response_func(source_name, batch_id, api_url, 200, 
                       response_time_ms=api_duration_ms)
        
        if not jobs_data:
            duration = time.time() - start_time
            complete_pipeline_run_func(batch_id, 'SUCCESS')
            log_audit_func(batch_id, pipeline_name, 'SUCCESS',
                          rows_read=0, rows_written=0, runtime_seconds=duration)
            print(f"  WARNING: No data returned")
            return True
        
        # Process to bronze layer
        print(f"\n[2/3] Processing to bronze layer...")
        bronze_records = process_to_bronze(jobs_data, source_name, batch_id, extract_job_id_func)
        records_processed = len(bronze_records)
        print(f"  Processed {records_processed} snapshots")
        
        # Write to bronze layer
        print(f"\n[3/3] Writing to {BRONZE_JOB_SNAPSHOT}...")
        df = spark.createDataFrame(bronze_records, schema=BRONZE_SNAPSHOT_SCHEMA)
        df.write.mode('append').saveAsTable(BRONZE_JOB_SNAPSHOT)
        print(f"  Wrote {records_processed} records")
        
        duration = time.time() - start_time
        
        # Complete pipeline run
        complete_pipeline_run_func(batch_id, 'SUCCESS')
        log_audit_func(batch_id, pipeline_name, 'SUCCESS',
                      rows_read=records_fetched, 
                      rows_written=records_processed, 
                      runtime_seconds=duration)
        
        print(f"\nSUCCESS - Ingested {records_processed} jobs in {duration:.2f}s")
        return True
        
    except Exception as e:
        duration = time.time() - start_time
        error_msg = str(e)
        complete_pipeline_run_func(batch_id, 'FAILED')
        log_audit_func(batch_id, pipeline_name, 'FAILED',
                      runtime_seconds=duration, error_message=error_msg)
        print(f"\nFAILED - {error_msg}")
        return False